# מחברת 4: CUDA building blocks של LLM

המטרה היא לעקוב אחרי token קטן דרך פעולות שמופיעות ב-Transformer. מתחילים ב-reference פשוט ב-Python, ואז מריצים kernels חינוכיים על A100.

## מפת tensors

```text
token_id: scalar
embedding table: [vocab_size, hidden_size]
hidden state: [hidden_size]
attention scores: [sequence_length, sequence_length]
projection weight: [hidden_size, output_size]
logits: [vocab_size]
```

במודל אמיתי יהיו גם batch, sequence, heads ו-head dimension. כאן משאירים כל shape קטן כדי לראות את הרעיון.

In [ ]:
from pathlib import Path
import math
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
print("repo:", ROOT)


## Reference קטן: token -> embedding -> residual -> RMSNorm

ה-reference אינו מחליף את CUDA. הוא נותן expected behavior שאפשר להשוות אליו.

In [ ]:
embeddings = [
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 2.0, 3.0, 4.0],
    [0.0, 0.0, 1.0, 0.0],
]
token_id = 1
residual_update = [0.5, 0.5, 0.5, 0.5]
hidden = embeddings[token_id][:]
hidden = [x + update for x, update in zip(hidden, residual_update)]
mean_square = sum(x * x for x in hidden) / len(hidden)
normalized = [x / math.sqrt(mean_square + 1e-5) for x in hidden]
print("token_id:", token_id)
print("embedding shape: [4], value:", embeddings[token_id])
print("after residual shape: [4], value:", hidden)
print("after RMSNorm shape: [4], value:", normalized)
assert len(normalized) == 4


## Causal mask ו-stable softmax

Query position יכול לראות רק את עצמו ואת העבר. ב-softmax מחסרים את המקסימום כדי למנוע overflow.

In [ ]:
sequence_length = 4
mask = [[0.0 if key <= query else -1e9 for key in range(sequence_length)]
        for query in range(sequence_length)]
for row in mask:
    print(row)

scores = [1000.0, 1001.0, 1002.0, 1003.0]
maximum = max(scores)
exps = [math.exp(score - maximum) for score in scores]
probabilities = [value / sum(exps) for value in exps]
print("softmax:", probabilities, "sum:", sum(probabilities))
assert abs(sum(probabilities) - 1.0) < 1e-12


## Build והרצת דוגמאות CUDA

התא הבא דורש A100, ‏CUDA Toolkit ו-CMake. הוא נכשל במפורש אם prerequisite חסר.

In [ ]:
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
print(subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True).stdout)
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "-j"], check=True)


In [ ]:
targets = [
    "llm_01_token_embedding",
    "llm_02_residual_add",
    "llm_03_silu_activation",
    "llm_04_rmsnorm",
    "llm_05_causal_mask",
    "llm_06_attention_softmax",
    "llm_07_linear_projection",
    "llm_08_mini_transformer_step",
]
for target in targets:
    result = subprocess.run([str(ROOT / "build" / target)], text=True, capture_output=True)
    print(target, "->", result.stdout.strip())
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0
    assert "PASS" in result.stdout


## מה להסביר בסיום

1. למה embedding lookup אינו softmax?
2. למה residual add הוא למעשה vector addition?
3. איפה RMSNorm זקוק ל-reduction ול-`__syncthreads()`?
4. למה causal mask הוא בעיה דו-ממדית?
5. למה מחסרים maximum ב-softmax?
6. למה linear projection הוא סוג של matrix-vector multiplication?
7. אילו פעולות production היה כדאי לבצע ב-fused kernel?